In [ ]:
'''# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session'''

### Introduction

In this experiment,  Simplified Graph Convolution (SGC) is implemented, that removes nonlinearities and collapses multiple graph convolution layers into a single linear transformation.

By precomputing feature propagation across the graph, SGC reduces model complexity while retaining the ability to capture structural information.

In [1]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.2 MB/s eta 0:00:00a 0:00:01


In [2]:
import torch
import torch_geometric

print(torch.__version__)
print(torch_geometric.__version__)

2.10.0+cu128
2.7.0


### Loading Amazon Computer Dataset

In [3]:
from torch_geometric.datasets import Amazon
dataset=Amazon(root='/kaggle/working/Amazon',name='Computers')
data=dataset[0]
print(data)


Processing...


Data(x=[13752, 767], edge_index=[2, 491722], y=[13752])


Done!


In [4]:
print(data.x.shape)
print(data.edge_index.shape)
print(data.y.shape)
print(data.y.unique())

torch.Size([13752, 767])
torch.Size([2, 491722])
torch.Size([13752])
tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])


### Loading Train Test Validation  Split (70% train 15% test 15% val)

In [6]:
from sklearn.model_selection import train_test_split
import numpy as np
y=data.y.cpu().numpy()
indices=np.arange(len(y))
train_idx,temp_idx=train_test_split(indices,stratify=y,test_size=0.3)
val_idx,test_idx=train_test_split(temp_idx,stratify=y[temp_idx],test_size=0.5)
train_idx=torch.tensor(train_idx)
val_idx=torch.tensor(val_idx)
test_idx=torch.tensor(test_idx)


In [7]:
torch.save(train_idx, "/kaggle/working/train_idx.pt")
torch.save(val_idx, "/kaggle/working/val_idx.pt")
torch.save(test_idx, "/kaggle/working/test_idx.pt")

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_idx = torch.load("/kaggle/input/datasets/mrxn251/gnn-splits/train_idx.pt")
val_idx = torch.load("/kaggle/input/datasets/mrxn251/gnn-splits/val_idx.pt")
test_idx = torch.load("/kaggle/input/datasets/mrxn251/gnn-splits/test_idx.pt")
train_idx = train_idx.to(device)
val_idx = val_idx.to(device)
test_idx = test_idx.to(device)

In [12]:
print(len(train_idx), len(val_idx), len(test_idx))

9626 2063 2063


In [26]:
#moving data to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(device)


### Defining SGC model architecture 

In [41]:
from torch_geometric.nn import SGConv
import torch.nn.functional as F

class SGC(torch.nn.Module):
    def __init__(self,in_channels,out_channels,K=2):
        super().__init__()
        self.conv=SGConv(in_channels,out_channels,K=K,cached=True)
    def forward(self,x,edge_index):
        return self.conv(x,edge_index)

### Training Function

In [50]:
def train_sgc(model,data,train_idx,optimizer,device):
    model.train()
    optimizer.zero_grad()
    out=model(data.x,data.edge_index)
    loss=F.cross_entropy(out[train_idx],data.y[train_idx])
    loss.backward()
    optimizer.step()
    return loss.item()
    

### Evaluation Function

In [52]:
@torch.no_grad()
def evaluate_sgc(model, data, val_idx):
    model.eval()

    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)

    correct = (pred[val_idx] == data.y[val_idx]).sum().item()
    total = len(val_idx)

    return correct / total

### Testing Function

In [96]:
from sklearn.metrics import f1_score


@torch.no_grad()
def test_sgc_with_f1(model, data, test_idx, device):
    model.eval()

    data = data.to(device)

    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)

    
    pred = pred[test_idx]
    labels = data.y[test_idx]

    pred = pred.cpu().numpy()
    labels = labels.cpu().numpy()

    acc = (pred == labels).mean()
    macro_f1 = f1_score(labels, pred, average='macro')
    micro_f1 = f1_score(labels, pred, average='micro')

    return acc, macro_f1, micro_f1

## Experiment Pipeline

This function manages the end-to-end training and evaluation process.

- Initializes the model using the given configuration
- Constructs NeighborLoaders for mini-batch training
- Trains the model across epochs with early stopping based on validation performance
- Saves the best-performing model
- Evaluates final performance on the test set using Accuracy and F1 scores

In [87]:
import random
import numpy as np
def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [99]:
def run_sgc_experiment(config, seed=0, run_test=False):
    set_seed(seed)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    epochs = 50 if not run_test else 100

    model = SGC(
        data.num_features,
        dataset.num_classes,
        K=config["K"]
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])

    data_device = data.to(device)

    best_val = 0
    patience = 10
    counter = 0
    save_model = run_test

    for epoch in range(1, epochs + 1):

        #  TRAIN
        model.train()
        optimizer.zero_grad()

        out = model(data_device.x, data_device.edge_index)

        loss = F.cross_entropy(
            out[train_idx],
            data_device.y[train_idx]
        )

        loss.backward()
        optimizer.step()

        # VALIDATION
        model.eval()
        with torch.no_grad():
            out = model(data_device.x, data_device.edge_index)
            pred = out.argmax(dim=1)

            correct = (pred[val_idx] == data_device.y[val_idx]).sum().item()
            val_acc = correct / len(val_idx)

        #  EARLY STOPPING
        if val_acc > best_val:
            best_val = val_acc
            counter = 0

            if save_model:
                torch.save(
                    model.state_dict(),
                    f"/kaggle/working/best_modelSGC_{seed}.pt"
                )
        else:
            counter += 1

        if counter >= patience:
            break

    if not run_test:
        return best_val

    #  LOAD BEST MODEL
    model.load_state_dict(
        torch.load(f"/kaggle/working/best_modelSGC_{seed}.pt", map_location=device)
    )

    #  TEST
    acc, macro_f1, micro_f1 = test_sgc_with_f1(
        model,
        data,
        test_idx,
        device
    )

    return acc, macro_f1, micro_f1

In [88]:
configs = [
    {"lr": 0.01, "K": 2},
    {"lr": 0.01, "K": 3},
    {"lr": 0.005, "K": 2},
    {"lr": 0.005, "K": 3},
]



### Hyperparameter Tuning

In [89]:
best_val = 0
best_config = None

for config in configs:
    val_acc = run_sgc_experiment(config, seed=0, run_test=False)
    
    print(config, "→", val_acc)

    if val_acc > best_val:
        best_val = val_acc
        best_config = config

print("Best config:", best_config)

{'lr': 0.01, 'K': 2} → 0.8056228793019874
{'lr': 0.01, 'K': 3} → 0.7707222491517208
{'lr': 0.005, 'K': 2} → 0.8061076102762966
{'lr': 0.005, 'K': 3} → 0.7707222491517208
Best config: {'lr': 0.005, 'K': 2}


### Final Training across 10 different seeds

In [100]:
#Final Training
acc_list = []
micro_f1_list = []
macro_f1_list=[]
for seed in range(10):
    acc, macro_f1, micro_f1 = run_sgc_experiment(best_config, seed=seed, run_test=True)
    acc_list.append(acc)
    macro_f1_list.append(macro_f1)
    micro_f1_list.append(micro_f1)

    print(f"Seed {seed}: Acc={acc:.4f}, macro_F1={macro_f1:.4f},micro_F1={micro_f1:.4f}")



Seed 0: Acc=0.8628, macro_F1=0.8427,micro_F1=0.8628
Seed 1: Acc=0.8502, macro_F1=0.8193,micro_F1=0.8502
Seed 2: Acc=0.8570, macro_F1=0.8427,micro_F1=0.8570
Seed 3: Acc=0.8531, macro_F1=0.8351,micro_F1=0.8531
Seed 4: Acc=0.8483, macro_F1=0.8311,micro_F1=0.8483
Seed 5: Acc=0.8585, macro_F1=0.8430,micro_F1=0.8585
Seed 6: Acc=0.8546, macro_F1=0.8337,micro_F1=0.8546
Seed 7: Acc=0.8589, macro_F1=0.8398,micro_F1=0.8589
Seed 8: Acc=0.8531, macro_F1=0.8424,micro_F1=0.8531
Seed 9: Acc=0.8488, macro_F1=0.8290,micro_F1=0.8488


### Final results

In [101]:
import numpy as np

acc_mean = np.mean(acc_list)
acc_std = np.std(acc_list)

macro_f1_mean = np.mean(macro_f1_list)
macro_f1_std = np.std(macro_f1_list)

micro_f1_mean = np.mean(micro_f1_list)
micro_f1_std = np.std(micro_f1_list)



print(f"\nFinal Results:")
print(f"Accuracy: {acc_mean:.4f} ± {acc_std:.4f}")
print(f"Macro F1: {macro_f1_mean:.4f} ± {macro_f1_std:.4f}")
print(f"Micro F1: {micro_f1_mean:.4f} ± {micro_f1_std:.4f}")



Final Results:
Accuracy: 0.8545 ± 0.0045
Macro F1: 0.8359 ± 0.0074
Micro F1: 0.8545 ± 0.0045
